# BaTiO₃ valence and conduction band effective masses

`analysis.effective_mass` locates a Wannier Hamiltonian's band extrema and computes effective-mass tensors from the analytic Fourier-interpolated Hessian — no finite-difference DFT band structure needed once Wannierized. This notebook applies it to cubic BaTiO₃'s valence-band maximum (VBM, O-2$p$) and conduction-band minimum (CBM, Ti-$t_{2g}$), reusing the exact same 12-MLWF Wannierisation as notebook 02 (9 O-2$p$ valence + 3 Ti-$t_{2g}$ conduction, genuinely disentangled) — the natural candidate for this since it already spans both band edges in one Wannier Hamiltonian.

In [1]:
import os, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

HERE = pathlib.Path.cwd()
REPO = HERE
while not (REPO / 'waw').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import waw
from waw.interfaces import quantum_espresso as qe   # the direct-input QE driver
from waw.interfaces.ase.driver import wannierize
from waw.units import BOHR_TO_ANG, HARTREE_TO_EV, EV_TO_HARTREE
from waw.vis import plot_bands, BandSeries

PSEUDO_DIR = REPO / 'workflows' / 'pseudos'   # shared across workflows/, not notebooks/-specific
NCORES = 16
waw.set_num_threads(NCORES)
print('waw', 'threads =', waw.get_num_threads(), '| repo', REPO)

waw threads = 16 | repo /aims_data/miguel/nas-data001/claude/wannier


### 1. Structure and converged DFT — identical recipe to notebook 02

In [2]:
from ase import Atoms
from waw.interfaces.ase.structure import real_lattice, recip_lattice
from waw.interfaces.projections import spd_projections
from waw.analysis.effective_mass import (
    semiconductor_band_edges, degenerate_effective_mass, masses_along)

a = 7.44266 * BOHR_TO_ANG
frac = [[0, 0, 0], [0.5, 0.5, 0.5], [0, 0.5, 0.5], [0.5, 0.5, 0], [0.5, 0, 0.5]]
atoms = Atoms('BaTiO3', scaled_positions=frac,
              cell=[[a, 0, 0], [0, a, 0], [0, 0, a]], pbc=True)
MP_GRID = (4, 4, 4)
WORK = HERE / 'runs' / 'batio3'

o_sites = [(0, 0.5, 0.5), (0.5, 0.5, 0), (0.5, 0, 0.5)]
projections = [p for o in o_sites for p in spd_projections(o, 'p')]
t2g_mr = (2, 3, 5)   # wannier90 mr order for l=2: dz2,dxz,dyz,dx2-y2,dxy
projections += [((0.5, 0.5, 0.5), 2, mr, 1, (0., 0., 1.), (1., 0., 0.), 1.0)
                for mr in t2g_mr]

ov = qe.generate_overlaps(
    atoms, MP_GRID, WORK, 'batio3',
    ecutwfc=60, scf_kpts=(8, 8, 8), nbnd=26, num_wann=12,
    exclude_bands=list(range(1, 12)),   # Ba/Ti/O semicore + O-2s
    projections=projections,
    pseudopotentials={'Ba': 'Ba.upf', 'Ti': 'Ti.upf', 'O': 'O.upf'},
    pseudo_dir=PSEUDO_DIR, ncores=NCORES,
    rerun_scf=False, rerun_nscf=False,   # reuse notebook 02's completed run
)
result = wannierize(
    atoms, MP_GRID, ov['kpts'],
    mmn=ov['mmn'], amn=ov['amn'], eig=ov['eig'],
    nnkpts=ov['nnkpts'], g_vectors=ov['g_vectors'], nw=12,
    frozen_window=(-1e3, 12.0), outer_window=(-1e3, 16.0),
    n_restarts=3, dis_n_iter=1000, n_iter=3000, verbose=False,
)
print(f'Omega_I = {result.dis.omega_i * BOHR_TO_ANG**2:.4f} Ang^2')
print(f'Omega_total = {result.omega_final * BOHR_TO_ANG**2:.4f} Ang^2')

Omega_I = 13.7310 Ang^2
Omega_total = 14.0209 Ang^2


### Interpolated bands against the ab-initio reference
The Wannier bands (lines) over a non-self-consistent `calculation='bands'` run
along the same path (points), with the disentanglement windows shaded and
everything referred to $E_F$. Inside the frozen window the interpolation must
reproduce the ab-initio bands exactly; the deviation printed on the
Wannierization mesh is the quantitative form of the same statement.

In [ ]:
from waw.interfaces.ase.structure import band_path
from waw.analysis.elph import band_eigensystem
from waw.interfaces.quantum_espresso import bands_along_path
from waw.analysis.bands import mesh_fidelity
from waw.vis import plot_wannierization_windows

E_F = ov['fermi_energy']
bp = band_path(atoms, npoints=150)
xcoords, xspecial, labels = bp.get_linear_kpoint_axis()
bands = band_eigensystem(result.hr, bp.kpts)[0] * HARTREE_TO_EV
try:
    dft_bands = bands_along_path(WORK, 'batio3', bp.kpts, ncores=NCORES)
except FileNotFoundError as exc:      # charge density not kept in this run dir
    dft_bands = np.full_like(bands, np.nan)
    print(exc)

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
plot_wannierization_windows(
    ax, xcoords, dft_bands, bands,
    outer_window=(-1e3, 16.0), frozen_window=(-1e3, 12.0), fermi_energy=E_F,
    xticks=xspecial, xticklabels=[l.replace('G', r'$\Gamma$') for l in labels])
ax.set_title('BaTiO$_3$: DFT (points) vs Wannier interpolation (lines)')
plt.tight_layout(); plt.show()

fid = mesh_fidelity(result.hr, ov['kpts'], ov['eig'],
                    window=(-1e3, 12.0), fermi_ev=E_F)
print(f"on the Wannier mesh, inside the frozen window: max {fid['max']:.3f} meV, "
      f"rms {fid['rms']:.3f} meV over {fid['n_states']} states")


### 2. Band edges and effective masses
`semiconductor_band_edges(hr, n_valence=9, ...)` searches band index 8 (VBM) and 9 (CBM) — the 9 O-2$p$ orbitals are fully occupied, the 3 Ti-$t_{2g}$ orbitals empty — over a coarse mesh, refines each extremum, and reports the effective-mass tensor from the analytic Hessian at each (a single-band treatment; see the degenerate cross-check below for extrema with band-touching neighbours).

In [4]:
recip = recip_lattice(atoms)
edges = semiconductor_band_edges(result.hr, n_valence=9, recip_lattice=recip,
                                 mesh=(20, 20, 20))
print(f'gap = {edges.gap * HARTREE_TO_EV:.4f} eV, direct = {edges.direct}')
print()
for v in edges.vbm:
    print(f'VBM at {np.round(v.extremum.kpt, 3)}, '
          f'E = {v.extremum.energy * HARTREE_TO_EV:.4f} eV, '
          f'principal masses (m_e) = {np.round(v.principal_masses, 3)}, '
          f'degenerate with bands {v.extremum.degenerate_with}')
for c in edges.cbm:
    print(f'CBM at {np.round(c.extremum.kpt, 3)}, '
          f'E = {c.extremum.energy * HARTREE_TO_EV:.4f} eV, '
          f'principal masses (m_e) = {np.round(c.principal_masses, 3)}, '
          f'degenerate with bands {c.extremum.degenerate_with}')

gap = 1.9114 eV, direct = False

VBM at [0.5 0.5 0.5], E = 11.1095 eV, principal masses (m_e) = [0.832 1.075 1.738], degenerate with bands (6, 7)
VBM at [0. 1. 0.], E = 11.0126 eV, principal masses (m_e) = [0.655 0.897 1.021], degenerate with bands (6, 7)
CBM at [0. 0. 1.], E = 13.0210 eV, principal masses (m_e) = [3.549 4.475 5.266], degenerate with bands (10, 11)


**The R→Γ indirect gap.** The VBM sits at $(\tfrac12,\tfrac12,\tfrac12)$ — the cubic perovskite BZ's $R$ point — and the CBM at $(0,0,1)\equiv\Gamma$ (the same point modulo a reciprocal lattice vector): the well-known $R\to\Gamma$ indirect gap of ideal cubic BaTiO₃. The O-2$p$ VBM masses come out light ($\sim$0.8-1.7 $m_e$); the Ti-$t_{2g}$ CBM masses are noticeably heavier ($\sim$3.5-5.3 $m_e$), consistent with the real, well-known physics that $t_{2g}$ conduction bands in perovskite titanates are narrow (limited direct $d$-$d$ overlap) compared to the wider $e_g$ manifold.

### 3. Degenerate (Löwdin k·p) cross-check
Both extrema are 3-fold degenerate (O-2$p$/Ti-$t_{2g}$ triplets), so the single-band Hessian above mixes the group via the sorted-eigenvalue assignment. `degenerate_effective_mass` does the proper quasi-degenerate perturbation-theory treatment against the remaining ("remote") bands instead.

In [5]:
for label, ext_list in [('VBM', edges.vbm), ('CBM', edges.cbm)]:
    e = ext_list[0].extremum
    if not e.degenerate_with:
        continue
    deg = degenerate_effective_mass(result.hr, e, recip)
    for direction, name in [([1, 0, 0], '[100]'), ([1, 1, 0], '[110]'), ([1, 1, 1], '[111]')]:
        print(f'{label} masses along {name} (m_e) =', np.round(masses_along(deg, direction), 3))

VBM masses along [100] (m_e) = [ 0.746  0.747 10.487]
VBM masses along [110] (m_e) = [0.742 1.035 2.132]
VBM masses along [111] (m_e) = [0.912 0.913 1.672]
CBM masses along [100] (m_e) = [0.372 0.373 5.673]
CBM masses along [110] (m_e) = [0.372 0.603 0.831]
CBM masses along [111] (m_e) = [0.5   0.5   0.647]


**All masses come out positive**, as they must at a genuine extremum (a real VBM/CBM curves the same way — down/up — in every direction). Both triplets show real, direction-dependent band warping the single-band Hessian estimate above cannot capture (it mixes the degenerate group via the sorted-eigenvalue assignment instead of the proper quasi-degenerate treatment): the VBM has one distinctly heavier branch along $[100]$ ($\sim$10.5 $m_e$ vs. $\sim$0.75 $m_e$ for the other two — a heavy-hole/light-hole-like split), and the CBM is markedly lighter along $[111]$ ($\sim$0.5 $m_e$) than the single-band estimate's isotropic-looking $\sim$3.5-5.3 $m_e$ suggested — direction-resolved information the naive treatment simply can't provide for a genuinely degenerate extremum.

